In [8]:
import os, json, re
import pandas as pd
from tqdm import tqdm
import ollama
import torch

In [9]:
# Define device (just for consistency, no specific changes needed for Ollama)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# -------------------------------
# 1️⃣ Model wrapper for Ollama
# -------------------------------
class CatalogLLMExtractor:
    def __init__(self, model_name="llama3.1:8b", device=device):
        """
        Initialize the CatalogLLMExtractor to interact with Ollama models.
        """
        self.device = device
        self.model_name = model_name
        # Make sure to specify the model name correctly in Ollama
        print(f"Using Ollama model: {self.model_name}")

    def build_prompt(self, catalog_content):
        return f"""
You are an information extraction assistant.

Given the raw catalog text of an e-commerce product, extract and return the following information as per the output format below.
- product name (string)  --> product name without the brand name and quantity details.
- brand (string or null)
- category (string or null)
- subcategory (string or null)
- flavor (string or null)
- value (number or null)
- unit (string or null)  --> Always use full unit names such as "gram", "kilogram", "millilitre", "litre", "fluid ounce", "ounce", "pound", "count", etc. Do not use abbreviations.
- pack_count (number or null)
- is_gluten_free (true or false)  --> true if the product is explicitly gluten-free, false otherwise.
- is_organic (true or false)  --> true if the product is explicitly organic, false otherwise.
- gender_target (male/female/unisex/null)
- packaging (string or null)
- contains_allergens --> string of allergens separated by comma, if none, return "none".
- notes (string or null) → add any uncertainty or comment if needed.

Rules:
- Output must not have any explanations.
- Derive brand and category logically from the text.
- Infer gender_target from product cues (e.g. 'men', 'women', 'kids').
- If units or values appear (like '12 oz' or 'Pack of 6'), extract them.

Example Output:
item_name: La Victoria Green Taco Sauce Mild, 12 Ounce (Pack of 6),
bullet_point_summary: Mild green taco sauce ideal for Mexican cuisine.,
brand: La Victoria,
category: condiment,
subcategory: sauce,
flavor: green taco,
value: 12,
unit: ounce,
pack_count: 6,
is_gluten_free: false,
is_organic: false,
gender_target: unisex,
packaging: bottle,
contains_allergens: none
notes: none

Now extract from the following raw catalog content:
---
{catalog_content}
---
"""

    def extract_one(self, catalog_text: str):
        # print(catalog_text)
        prompt = self.build_prompt(catalog_text)
        response = ollama.chat(model=self.model_name, messages=[{"role": "user", "content": prompt}])
        result = response["message"]["content"]
        return result


In [11]:
# -------------------------------
# 2️⃣ Batch extraction
# -------------------------------
# create a new dataframe with the extracted information
def run_llm_extraction(df, model_name="llama3.1:8b", device=device):
    """Run extraction for a dataframe using Ollama."""
    extractor = CatalogLLMExtractor(model_name, device=device)
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        catalog_text = str(row["catalog_content"])
        extracted_info = extractor.extract_one(catalog_text)
        results.append(catalog_text + '\n' + extracted_info)
    return results

In [12]:
# -------------------------------
# 3️⃣ Usage
# -------------------------------
df = pd.read_csv("../dataset/train.csv")
# Compute additional engineered features
df["catalog_word_len"] = df["catalog_content"].astype(str).apply(lambda x: len(x.split()))


In [ ]:
# Run extraction for the full dataset
extracted_results = run_llm_extraction(df, model_name="llama3.1:8b", device=device)

In [ ]:
# save to the dataframe as new column
df["llm_extracted_info"] = extracted_results
df.to_csv("../dataset/train_with_llm_extraction_ollama.csv", index=False)